# 随机作业车间调度问题

**类别：** 调度

来源: [https://www.hexaly.com/templates/stochastic-job-shop-scheduling-problem](https://www.hexaly.com/templates/stochastic-job-shop-scheduling-problem)


## 问题

**在作业车间调度问题中**，一组作业必须在车间中的每台机器上完成加工。每个作业由若干按顺序排列的任务（称为活动）组成。一个活动表示该作业在某台机器上的加工过程，并具有给定的加工时间。每个作业在每台机器上都有一个活动，且每个活动只能在其前一个活动结束之后才能开始。每台机器同一时刻只能处理一个活动。在本例中，我们考虑带有多场景的作业车间调度问题的随机版本：每个活动的加工时间在不同场景下会有所不同。对于给定的作业加工顺序和给定的场景，makespan 是所有作业加工完成的时间。本问题的目标是寻找一种作业顺序，使得所有场景下的最大 makespan 最小化。

	

### 学到的建模原则

- 添加 [interval 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模各活动
- 添加 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模每台机器上活动的加工顺序
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 将 interval 变量和 list 变量联系起来


## 数据

数据文件的格式如下：

- 第一行：作业数、机器数、场景数。
- 从第四行开始，对每个场景：

- 对每个作业：依次给出其在该作业处理顺序中各机器上的加工时间。
- 对每个作业：给出其加工顺序（即按访问顺序排列的机器列表，所有场景都相同）。


## 模型

随机作业车间调度问题的 Hexaly 模型使用 interval 决策变量来建模各场景下各活动的时间区间。我们将每个 interval 的长度约束为相应场景下该活动的加工时间。然后可以写出紧前约束：对于每个作业以及每个场景，该作业的每个活动必须在其前一台机器上的活动结束之后才能开始。

除了表示各活动时间区间的 interval 决策变量外，我们还使用 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。一个 list 用于建模一台机器上各活动的加工顺序。使用 **count（计数）** 运算符约束列表的大小，可以确保每台机器处理每个作业。析取资源约束——每台机器同一时刻只能处理一个活动——可表述为：对任意 i，位置 i+1 上处理的活动必须在位置 i 上处理的活动结束后才能开始。为建模这些约束，我们将 interval 决策（时间区间）与 list 决策（作业顺序）配对起来。我们编写一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来表达两个相邻活动之间的关系。该函数在每个场景中、对每台机器处理的所有活动通过可变参数数量的 **and（与）** 运算符组合使用。

目标是使所有场景中的最大 makespan 最小化，对每个场景而言该值即为所有活动都加工完成时的时间。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


def read_instance(filename):
    with open(filename) as f:
        lines = f.readlines()

    first_line = lines[1].split()
    # Number of jobs
    nb_jobs = int(first_line[0])
    # Number of machines
    nb_machines = int(first_line[1])
    # Number of scenarios
    nb_scenarios = int(first_line[2])

    # Processing times for each job on each machine (given in the processing order)
    processing_times_in_processing_order_per_scenario = [[[int(lines[s*(nb_jobs+1)+i].split()[j])
                                                           for j in range(nb_machines)]
                                                          for i in range(3, 3 + nb_jobs)]
                                                         for s in range(nb_scenarios)]

    # Processing order of machines for each job
    machine_order = [[int(lines[i].split()[j]) - 1 for j in range(nb_machines)]
                     for i in range(4 + nb_scenarios*(nb_jobs+1), 4 + nb_scenarios*(nb_jobs+1) + nb_jobs)]

    # Reorder processing times: processing_time[s][j][m] is the processing time of the
    # task of job j that is processed on machine m in the scenario s
    processing_time_per_scenario = [[[processing_times_in_processing_order_per_scenario[s][j][machine_order[j].index(m)]
                                      for m in range(nb_machines)]
                                     for j in range(nb_jobs)]
                                    for s in range(nb_scenarios)]

    # Trivial upper bound for the end times of the tasks
    max_end = max([sum(sum(processing_time_per_scenario[s][j])
                    for j in range(nb_jobs)) for s in range(nb_scenarios)])

    return nb_jobs, nb_machines, nb_scenarios, processing_time_per_scenario, machine_order, max_end


def main(instance_file, output_file, time_limit):
    nb_jobs, nb_machines, nb_scenarios, processing_time_per_scenario, machine_order, max_end = read_instance(
        instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Interval decisions: time range of each task
        # tasks[s][j][m] is the interval of time of the task of job j which is processed
        # on machine m in the scenario s
        tasks = [[[model.interval(0, max_end) for m in range(nb_machines)]
                  for j in range(nb_jobs)]
                 for s in range(nb_scenarios)]

        # Task duration constraints
        for s in range(nb_scenarios):
            for j in range(nb_jobs):
                for m in range(0, nb_machines):
                    model.constraint(model.length(tasks[s][j][m]) == processing_time_per_scenario[s][j][m])

        # Create an Hexaly array in order to be able to access it with "at" operators
        task_array = model.array(tasks)

        # Precedence constraints between the tasks of a job
        for s in range(nb_scenarios):
            for j in range(nb_jobs):
                for k in range(nb_machines - 1):
                    model.constraint(
                        tasks[s][j][machine_order[j][k]] < tasks[s][j][machine_order[j][k + 1]])

        # Sequence of tasks on each machine
        jobs_order = [model.list(nb_jobs) for m in range(nb_machines)]

        for m in range(nb_machines):
            # Each job has a task scheduled on each machine
            sequence = jobs_order[m]
            model.constraint(model.eq(model.count(sequence), nb_jobs))

            # Disjunctive resource constraints between the tasks on a machine
            for s in range(nb_scenarios):
                sequence_lambda = model.lambda_function(
                    lambda i: model.lt(model.at(task_array, s, sequence[i], m),
                                       model.at(task_array, s, sequence[i + 1], m)))
                model.constraint(model.and_(model.range(0, nb_jobs - 1), sequence_lambda))

        # Minimize the maximum makespan: end of the last task of the last job
        # over all scenarios
        makespans = [model.max([model.end(tasks[s][j][machine_order[j][nb_machines - 1]]) for j in range(nb_jobs)])
                     for s in range(nb_scenarios)]
        max_makespan = model.max(makespans)
        model.minimize(max_makespan)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - for each machine, the job sequence
        #
        if output_file != None:
            final_jobs_order = [list(jobs_order[m].value) for m in range(nb_machines)]
            with open(output_file, "w") as f:
                print("Solution written in file ", output_file)
                for m in range(nb_machines):
                    for j in range(nb_jobs):
                        f.write(str(final_jobs_order[m][j]) + " ")
                    f.write("\n")


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python stochastic_jobshop.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
